In [1]:
import ollama

In [2]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": "What is the main problem in this review: "
                       "'The app crashes every time I try to open it.'"
        }
    ]
)

print(response["message"]["content"])

The main problem in this review is that the app consistently crashes when being opened. This indicates there might be significant issues with the app's stability and functionality, which could include programming errors or compatibility problems.


In [3]:
import pandas as pd

eval_df = pd.read_csv("../data/eval/eval_set.csv")

eval_df.head()

,feedback_id,rating,review_title,review_text,feedback,is_source_artifact,is_low_information,is_usable_feedback,text_for_ai,clean_title,gold_sentiment,gold_intent,gold_product_area,gold_theme,gold_severity,gold_actionability
0,52581,1.0,works for 20 seconds then crashes have to res...,works for 20 seconds then crashes. then you ha...,works for 20 seconds then crashes have to res...,False,False,True,works for 20 seconds then crashes have to res...,works for 20 seconds then crashes have to res...,negative,bug_report,reliability,App crashes repeatedly,high,high
1,33251,5.0,Great,I like this app i like a lot i really really d...,Great. I like this app i like a lot i really r...,False,False,True,Great I like this app i like a lot i really re...,Great,mixed,complaint,features,Annoying questionnaires,medium,high
2,50176,5.0,awesome,Awesome I hope it.will be awesome,awesome. Awesome I hope it.will be awesome,False,False,True,awesome Awesome I hope it.will be awesome,awesome,positive,praise,other,General satisfaction,low,low
3,32077,5.0,Great,Easy to save. Easy to use. Great recipes.,Great. Easy to save. Easy to use. Great recipes.,False,False,True,Great Easy to save. Easy to use. Great recipes.,Great,positive,praise,usability,Easy recipe management,low,medium
4,72436,1.0,Not worth it,Only able to use if you have a paid subscriber...,Not worth it. Only able to use if you have a p...,False,False,True,Not worth it Only able to use if you have a pa...,Not worth it,negative,pricing_feedback,pricing,Paid subscription required,high,high


In [4]:
test_review = eval_df[
    eval_df["feedback_id"] == 52581
]["text_for_ai"].iloc[0]

print(test_review)

works for 20 seconds then crashes  have to restart  useless works for 20 seconds then crashes. then you have to restart it for another 20 second view. not very encourragedbfo get the pro model.


In [5]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this customer review.

Customer feedback:

{test_review}

Tell me:
1. Sentiment
2. Main problem
3. Product area
4. Severity
5. Whether the company should take action
"""
        }
    ]
)

print(response["message"]["content"])

Certainly, let's analyze this customer review:

1. **Sentiment**: The sentiment of this review is negative. The customer expresses frustration and dissatisfaction with the product.

2. **Main Problem**: 
   - The main problem identified by the customer is that the product crashes after only 20 seconds of use.

3. **Product Area**:
   - This review pertains to a software or application, given the context of crashing after short usage periods.

4. **Severity**:
   - The severity can be considered moderate-severe due to the repetitive need for restarting and the limited functionality. A 20-second usable period is quite brief, which significantly impacts user experience.

5. **Whether the Company Should Take Action**:
   - Yes, the company should take action based on this feedback. The issues reported by the customer indicate a significant usability problem that needs to be addressed promptly.

In summary:
- Sentiment: Negative
- Main Problem: Crashing after 20 seconds of use, requiring fr

In [6]:
SENTIMENTS = [
    "positive",
    "negative",
    "neutral",
    "mixed"
]

INTENTS = [
    "bug_report",
    "complaint",
    "praise",
    "feature_request",
    "pricing_feedback",
    "question",
    "other"
]

PRODUCT_AREAS = [
    "reliability",
    "usability",
    "performance",
    "features",
    "pricing",
    "compatibility",
    "account",
    "other"
]

SEVERITIES = [
    "low",
    "medium",
    "high"
]

ACTIONABILITY = [
    "low",
    "medium",
    "high"
]

In [7]:
SYSTEM_PROMPT = """
You are a Voice-of-Customer analysis system for a product team.

Your job is to analyze customer feedback and classify it using ONLY
the allowed categories provided below.

SENTIMENT:
- positive
- negative
- neutral
- mixed

INTENT:
- bug_report
- complaint
- praise
- feature_request
- pricing_feedback
- question
- other

PRODUCT AREA:
- reliability
- usability
- performance
- features
- pricing
- compatibility
- account
- other

SEVERITY:
- low
- medium
- high

ACTIONABILITY:
- low
- medium
- high

Rules:

1. Use only the allowed labels.
2. Do not invent categories.
3. Base your analysis only on the customer feedback.
4. Do not assume information that isn't present.
5. theme should be a short, specific description of the customer's issue
   or positive experience.
6. evidence must be directly supported by the review.
7. confidence must be between 0 and 1.
8. If the evidence is unclear, lower the confidence.
9. Do not fabricate product behavior, causes, or customer information.
"""

In [8]:
import json

prompt = f"""
Analyze this customer review.

Customer feedback:
{test_review}

Return ONLY valid JSON with exactly these fields:

{{
    "sentiment": "...",
    "intent": "...",
    "product_area": "...",
    "theme": "...",
    "severity": "...",
    "actionability": "...",
    "confidence": 0.0,
    "evidence": "..."
}}
"""

In [13]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

```json
{
    "sentiment": "negative",
    "intent": "bug_report",
    "product_area": "reliability",
    "theme": "crashing every 20 seconds",
    "severity": "high",
    "actionability": "high",
    "confidence": 0.9,
    "evidence": "works for 20 seconds then crashes, have to restart"
}
```


In [15]:
import ollama

response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

raw_output = response["message"]["content"]

print(raw_output)

```json
{
    "sentiment": "negative",
    "intent": "bug_report",
    "product_area": "reliability",
    "theme": "crashes frequently",
    "severity": "high",
    "actionability": "high",
    "confidence": 0.9,
    "evidence": "works for 20 seconds then crashes, have to restart it"
}
```


In [16]:
import json
import re

def parse_json_output(raw_output):

    raw_output = raw_output.strip()

    # Remove markdown code fences if present
    raw_output = re.sub(r"^```json\s*", "", raw_output)
    raw_output = re.sub(r"^```\s*", "", raw_output)
    raw_output = re.sub(r"\s*```$", "", raw_output)

    try:
        return json.loads(raw_output)

    except json.JSONDecodeError:
        # Try extracting the JSON object
        match = re.search(r"\{.*\}", raw_output, re.DOTALL)

        if match:
            return json.loads(match.group())

        raise ValueError(
            "Could not find valid JSON in model output."
        )

In [17]:
result = parse_json_output(raw_output)

result

{'sentiment': 'negative',
 'intent': 'bug_report',
 'product_area': 'reliability',
 'theme': 'crashes frequently',
 'severity': 'high',
 'actionability': 'high',
 'confidence': 0.9,
 'evidence': 'works for 20 seconds then crashes, have to restart it'}

In [18]:
result = parse_json_output(raw_output)
print(result)

{'sentiment': 'negative', 'intent': 'bug_report', 'product_area': 'reliability', 'theme': 'crashes frequently', 'severity': 'high', 'actionability': 'high', 'confidence': 0.9, 'evidence': 'works for 20 seconds then crashes, have to restart it'}


In [21]:
def validate_output(result):

    errors = []

    if result["sentiment"] not in SENTIMENTS:
        errors.append(
            f"Invalid sentiment: {result['sentiment']}"
        )

    if result["intent"] not in INTENTS:
        errors.append(
            f"Invalid intent: {result['intent']}"
        )

    if result["product_area"] not in PRODUCT_AREAS:
        errors.append(
            f"Invalid product_area: {result['product_area']}"
        )

    if result["severity"] not in SEVERITIES:
        errors.append(
            f"Invalid severity: {result['severity']}"
        )

    if result["actionability"] not in ACTIONABILITY:
        errors.append(
            f"Invalid actionability: {result['actionability']}"
        )

    if not isinstance(result["confidence"], (int, float)):
        errors.append("Confidence must be numeric")

    elif not 0 <= result["confidence"] <= 1:
        errors.append("Confidence must be between 0 and 1")

    if not result["theme"].strip():
        errors.append("Theme is empty")

    if not result["evidence"].strip():
        errors.append("Evidence is empty")

    return errors

In [22]:
errors = validate_output(result)

print(errors)

[]


In [23]:
bad_result = {
    "sentiment": "very_negative",
    "intent": "bug_report",
    "product_area": "reliability",
    "theme": "app crashes",
    "severity": "critical",
    "actionability": "urgent",
    "confidence": 1.4,
    "evidence": ""
}

In [24]:
validate_output(bad_result)

['Invalid sentiment: very_negative',
 'Invalid severity: critical',
 'Invalid actionability: urgent',
 'Confidence must be between 0 and 1',
 'Evidence is empty']

In [25]:
def determine_review_status(result, errors):

    if errors:
        return "REJECT"

    if result["confidence"] < 0.70:
        return "HUMAN_REVIEW"

    if result["actionability"] == "high":
        return "HUMAN_REVIEW"

    return "AUTO_ACCEPT"

In [26]:
status = determine_review_status(result, errors)

print(status)

HUMAN_REVIEW


In [27]:
SYSTEM_PROMPT = """
You are a Voice-of-Customer analysis system for a product team.

Your job is to analyze customer feedback using ONLY the allowed categories.

SENTIMENT:
- positive
- negative
- neutral
- mixed

INTENT:
- bug_report
- complaint
- praise
- feature_request
- pricing_feedback
- question
- other

PRODUCT AREA:
- reliability
- usability
- performance
- features
- pricing
- compatibility
- account
- other

SEVERITY:
- low
- medium
- high

ACTIONABILITY:
- low
- medium
- high

Rules:

1. Use only the allowed labels.
2. Do not invent categories.
3. Use only information explicitly supported by the customer review.
4. Do not infer facts that are not present.
5. theme must describe the customer's actual issue or positive experience.
6. evidence must be a short exact quote or near-exact phrase from the review.
7. Never put information in evidence that does not appear in the review.
8. Do not use outside knowledge.
9. If uncertain, reduce confidence.
10. confidence must be between 0 and 1.
"""

In [28]:
test_review

'works for 20 seconds then crashes\xa0 have to restart\xa0 useless works for 20 seconds then crashes. then you have to restart it for another 20 second view. not very encourragedbfo get the pro model.'

In [29]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

raw_output = response["message"]["content"]

print(raw_output)

```json
{
    "sentiment": "negative",
    "intent": "bug_report",
    "product_area": "reliability",
    "theme": "crashing and instability",
    "severity": "high",
    "actionability": "high",
    "confidence": 0.9,
    "evidence": "works for 20 seconds then crashes"
}
```


In [52]:
import re

def normalize_text(text):
    text = str(text).lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    return text.strip()


def validate_evidence(result, review):

    evidence = normalize_text(result["evidence"])
    review = normalize_text(review)

    if not evidence:
        return False, "Evidence is empty"

    # 1. Exact normalized match
    if evidence in review:
        return True, "Evidence grounded: exact match"

    # 2. Token overlap check
    evidence_tokens = set(evidence.split())
    review_tokens = set(review.split())

    overlap = evidence_tokens.intersection(review_tokens)

    if len(evidence_tokens) == 0:
        return False, "Evidence is empty"

    overlap_ratio = len(overlap) / len(evidence_tokens)

    if overlap_ratio >= 0.80:
        return True, f"Evidence grounded: {overlap_ratio:.0%} token overlap"

    return False, "Evidence not sufficiently grounded in customer review"


In [53]:
evidence_ok, evidence_message = validate_evidence(
    result,
    test_review
)

print(evidence_ok)
print(evidence_message)

True
Evidence grounded: 100% token overlap


In [54]:
errors = run_guardrails(
    result,
    test_review
)

status = determine_review_status(
    result,
    errors
)

print("Errors:", errors)
print("Status:", status)

Errors: []
Status: HUMAN_REVIEW


In [55]:
errors = run_guardrails(
    result,
    test_review
)

print(errors)

[]


In [56]:
def determine_review_status(result, errors):

    if errors:
        return "REJECT"

    if result["confidence"] < 0.70:
        return "HUMAN_REVIEW"

    if result["severity"] == "high":
        return "HUMAN_REVIEW"

    if result["actionability"] == "high":
        return "HUMAN_REVIEW"

    return "AUTO_ACCEPT"

In [57]:
print(raw_output)
print(errors)
print(status)

```json
{
    "sentiment": "negative",
    "intent": "bug_report",
    "product_area": "reliability",
    "theme": "crashing and instability",
    "severity": "high",
    "actionability": "high",
    "confidence": 0.9,
    "evidence": "works for 20 seconds then crashes"
}
```
[]
HUMAN_REVIEW
